# Lab 3 — Solución (solo docente)

Referencia con parámetros recomendados y respuestas teóricas sugeridas.
Los alumnos trabajan solo con `xai_estructuras_alumno_ia.ipynb` + bitácora de prompts.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_xai, verificar_carga, verificar_features, verificar_limpieza,
    verificar_columna, verificar_etiquetas, verificar_xgboost, verificar_importancia_global,
    verificar_shap_global, verificar_shap_local, verificar_lime_local, verificar_pdp,
    resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración en tres ejes |
| Strain | με | Deformación (extensómetro) |
| Temp | °C | Temperatura del sensor |
| **Condition Label** | **0 / 1 / 2** | **Normal / daño menor / severo** |

Mismo CSV que **Lab 1**. Entrenamos **XGBoost** y aplicamos un **kit xAI**: importancia, permutation, **SHAP**, **LIME** y **PDP**.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama xAI en monitoreo estructural

El objetivo central del lab **no es solo entrenar** el clasificador: es **probar varias técnicas de explicación** sobre el mismo modelo.

### Caja de herramientas xAI (este lab)

| Técnica | Alcance | Sección | Idea clave |
|---------|---------|---------|------------|
| **Importancia del booster** | Global | 7 | Qué sensores usa el árbol en promedio |
| **Permutation importance** | Global | 7 (pre) | Cuánto cae el rendimiento si barajas una feature |
| **SHAP** (`TreeExplainer`) | Global + local | 8–9 | Contribución marginal por sensor (eficiente en árboles) |
| **LIME** | Local | 10 | Aproximación interpretable de **un caso** (comparar con SHAP) |
| **PDP + SHAP dependence** | Global marginal | 11 | Efecto promedio de un sensor vs interacciones |

### 📘 Subpreguntas
- **1.a** ¿Qué aporta xAI frente a solo mirar accuracy?
- **1.b** ¿Cuándo usarías SHAP vs LIME para explicar una alerta en obra?
- **1.c** ¿Qué diferencia hay entre explicación **global** y **local**?


#### ✅ Respuesta sugerida

**1.a** xAI muestra **qué sensores** empujaron la predicción; accuracy solo dice si acertó.
**1.b** SHAP es estable y rápido en árboles; LIME es flexible y útil para comparar un caso concreto con lenguaje simple.
**1.c** Global = promedio del modelo; local = una lectura de sensor en un instante.


In [ ]:
### TU TAREA AQUÍ ###
TECNICAS_XAI = ["importancia", "permutation", "shap", "lime", "pdp"]
print("Técnicas xAI del lab:")
for t in TECNICAS_XAI:
    print(f"  · {t}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `TECNICAS_XAI`
r = []
try:
    r = verificar_panorama_xai(TECNICAS_XAI)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama xAI', r)


## Pregunta 2 — Carga del dataset de sensores

### 📘 Subpreguntas
- **2.a** ¿Cuántas features de sensor vs 1 etiqueta de condición?
- **2.b** ¿Por qué excluimos `Timestamp` del modelo?


#### ✅ Reflexión 2

5 features + Timestamp + Condition Label. Timestamp ordena series pero no entra al modelo.


In [ ]:
# --- PRE-ESCRITO: carga ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


In [ ]:
### TU TAREA AQUÍ ###
FEATURES = [
    "Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)",
    "Strain (με)", "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features: {FEATURES}")
display(df.head(N_FILAS_HEAD))


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `FEATURES`, `N_FILAS_HEAD`
r = []
try:
    r = verificar_carga(df, N_FILAS_HEAD)
    r.extend(verificar_features(FEATURES))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Carga', r)


## Pregunta 3 — Calidad de datos y limpieza

### 📘 Subpreguntas
- **3.a** ¿Cuántas filas se pierden al eliminar nulos?
- **3.b** ¿Por qué `Strain` suele ser crítico en SHM?


#### ✅ Reflexión 3

96 filas eliminadas → 904. Strain es sensible al daño estructural.


In [ ]:
# --- PRE-ESCRITO: limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (crudo):")
display(n_nulos_por_col)
df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas")


In [ ]:
### TU TAREA AQUÍ ###
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas «{COLUMNA_REVISAR}» (crudo):")
display(stats_col)


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `COLUMNA_REVISAR`
r = []
try:
    r = verificar_limpieza(df_limpio, n_antes, n_despues)
    r.extend(verificar_columna(df, COLUMNA_REVISAR))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Limpieza', r)


## Pregunta 4 — Balance de clases (Condition Label)

### 📘 Subpreguntas
- **4.a** ¿La clase 0 es mayoritaria? ¿Qué implica para el modelo?
- **4.b** ¿Por qué usamos `stratify` en el split?


#### ✅ Reflexión 4

Clase 0 mayoritaria; stratify mantiene proporciones en train/test.


In [ ]:
# --- PRE-ESCRITO: distribución de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(conteo).plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_xlabel('Condition Label')
ax.set_ylabel('Conteo')
ax.set_title('Distribución de estados estructurales')
plt.tight_layout()
plt.show()
print("Conteo:", conteo)


In [ ]:
### TU TAREA AQUÍ ###
N_CLASES_MOSTRAR = 3
serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
display(serie_clases)


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `N_CLASES_MOSTRAR`
r = []
try:
    r = verificar_etiquetas(conteo, N_CLASES_MOSTRAR)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Balance de clases', r)


## Pregunta 5 — Partición y XGBoost

### 📘 Subpreguntas
- **5.a** ¿Por qué escalamos sensores antes de XGBoost?
- **5.b** ¿Qué controlan `max_depth` y `learning_rate`?


#### ✅ Reflexión 5

Escalado equilibra magnitudes (Accel_Z ~9.8 vs Strain ~80 με).


In [ ]:
# --- PRE-ESCRITO: escalado y split estratificado ---
X_raw = df_limpio[FEATURES].values
y = df_limpio['Condition Label'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


In [ ]:
### TU TAREA AQUÍ ###
N_ESTIMATORS = 100
MAX_DEPTH = 6
LEARNING_RATE = 0.1
modelo = XGBClassifier(
    objective='multi:softprob', num_class=3,
    n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE, random_state=RANDOM_STATE, eval_metric='mlogloss',
)
modelo.fit(X_train, y_train)
print("✅ XGBoost entrenado.")


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `N_ESTIMATORS`, `MAX_DEPTH`, `LEARNING_RATE`
r = []
try:
    y_pred_tmp = modelo.predict(X_test)
    acc_tmp = accuracy_score(y_test, y_pred_tmp)
    r = verificar_xgboost(acc_tmp, N_ESTIMATORS, MAX_DEPTH, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — XGBoost', r)


## Pregunta 6 — Métricas de clasificación

### 📘 Subpreguntas
- **6.a** ¿Qué clases se confunden más en la matriz?
- **6.b** ¿El recall de daño severo (clase 2) es aceptable para alertas?


#### ✅ Reflexión 6

Confusión 1↔2 es común; revisar recall de clase 2 para alertas.


In [ ]:
# --- PRE-ESCRITO: predicciones en test ---
y_pred = modelo.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)
print(f"Accuracy test: {acc_test:.3f}")


In [ ]:
### TU TAREA AQUÍ ###
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2], yticklabels=[0, 1, 2], ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión')
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `N_ESTIMATORS`, `MAX_DEPTH`, `LEARNING_RATE`
r = []
try:
    r = verificar_xgboost(acc_test, N_ESTIMATORS, MAX_DEPTH, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Métricas', r)


## Pregunta 7 — xAI global (1): importancia del booster y permutation

### 📘 Subpreguntas
- **7.a** ¿Coincide el top sensor con la física del daño (Strain)?
- **7.b** ¿Importancia del booster y permutation importance miden lo mismo?
- **7.c** ¿En qué se diferencian de SHAP (sección 8)?


#### ✅ Reflexión 7

Booster importance ≠ permutation: la segunda mide impacto en métrica al barajar.


In [ ]:
# --- PRE-ESCRITO: dos técnicas xAI globales ---
imp_series = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
imp_series.plot(kind='barh', ax=ax, color='#8e44ad')
ax.set_xlabel('Importancia (gain/weight)')
ax.set_title('xAI global — Importancia del booster (XGBoost)')
plt.tight_layout()
plt.show()

perm = permutation_importance(modelo, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE)
perm_series = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
perm_series.plot(kind='barh', ax=ax, color='#16a085')
ax.set_xlabel('Caída media en score al permutar')
ax.set_title('xAI global — Permutation importance')
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
TOP3_IMPORTANCIA = imp_series.sort_values(ascending=False).head(3).index.tolist()
print(f"Top-3: {TOP3_IMPORTANCIA}")


In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `TOP3_IMPORTANCIA`
r = []
try:
    r = verificar_importancia_global(TOP3_IMPORTANCIA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Importancia global', r)


## Pregunta 8 — xAI con SHAP global (summary plot)

### 📘 Subpreguntas
- **8.a** ¿Qué clase de daño explicas con `CLASE_SHAP`?
- **8.b** ¿Qué feature aparece con mayor |SHAP| en promedio?


#### ✅ Reflexión 8

Clase 2 = daño severo; summary muestra Strain con alto |SHAP|.


In [ ]:
# --- PRE-ESCRITO: TreeExplainer ---
def shap_values_clase(shap_values, clase: int):
    if isinstance(shap_values, list):
        return shap_values[clase]
    return shap_values[:, :, clase]

explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    n_clases_shap, n_casos_shap = len(shap_values), shap_values[0].shape[0]
else:
    n_casos_shap, _, n_clases_shap = shap_values.shape
print(f"SHAP: {n_clases_shap} clases × {n_casos_shap} casos test")


In [ ]:
### TU TAREA AQUÍ ###
CLASE_SHAP = 2
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
plt.title('SHAP summary — daño severo (clase 2)')
plt.tight_layout(); plt.show()


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `CLASE_SHAP`
r = []
try:
    r = verificar_shap_global(CLASE_SHAP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — SHAP global', r)


## Pregunta 9 — xAI local con SHAP (waterfall)

### 📘 Subpreguntas
- **9.a** ¿La predicción del caso coincide con la etiqueta real?
- **9.b** ¿Qué sensor empujó más la predicción en ese instante?
- **9.c** ¿Cómo se comparará con LIME en la sección 10?


#### ✅ Reflexión 9

Waterfall SHAP descompone una predicción; base para informes de inspección.


In [ ]:
# --- PRE-ESCRITO: utilidad para índice en test ---
def etiqueta_en_test(idx: int) -> tuple[int, int]:
    yt = int(y_test[idx])
    yp = int(y_pred[idx])
    return yt, yp


In [ ]:
### TU TAREA AQUÍ ###
INDEX_CASO = 0
y_true_caso, y_pred_caso = etiqueta_en_test(INDEX_CASO)
print(f"real={y_true_caso}, predicho={y_pred_caso}")
clase_explicar = CLASE_SHAP
sv = shap_values_clase(shap_values, clase_explicar)[INDEX_CASO]
base = explainer.expected_value
if isinstance(base, (list, np.ndarray)):
    base = base[clase_explicar]
exp = shap.Explanation(values=sv, base_values=base, data=X_test[INDEX_CASO], feature_names=FEATURES)
shap.plots.waterfall(exp, max_display=6, show=False)
plt.tight_layout(); plt.show()


In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `INDEX_CASO`, `y_true_caso`, `y_pred_caso`
r = []
try:
    r = verificar_shap_local(INDEX_CASO, len(X_test), y_true_caso, y_pred_caso)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — SHAP local', r)


## Pregunta 10 — xAI local con LIME (mismo caso que SHAP)

**LIME** (Local Interpretable Model-agnostic Explanations) aproxima el modelo **solo alrededor de un caso**.
Usa el mismo `INDEX_CASO` y `CLASE_SHAP` que en la sección 9 para **comparar** con el waterfall de SHAP.

### 📘 Subpreguntas
- **10.a** ¿LIME y SHAP coinciden en el sensor más influyente?
- **10.b** ¿Por qué LIME puede variar si repites la explicación?


#### ✅ Reflexión 10

LIME aproxima localmente; comparar con SHAP en el mismo INDEX_CASO.


In [ ]:
# --- PRE-ESCRITO: explainer LIME tabular ---
explainer_lime = LimeTabularExplainer(
    X_train,
    feature_names=FEATURES,
    class_names=['Normal (0)', 'Daño menor (1)', 'Daño severo (2)'],
    mode='classification',
    random_state=RANDOM_STATE,
)
print("✅ LIME TabularExplainer listo (fondo = train).")


In [ ]:
### TU TAREA AQUÍ ###
exp_lime = explainer_lime.explain_instance(
    X_test[INDEX_CASO], modelo.predict_proba, num_features=5, labels=(CLASE_SHAP,),
)
fig = exp_lime.as_pyplot_figure(label=CLASE_SHAP)
plt.title(f'LIME — caso {INDEX_CASO}')
plt.tight_layout(); plt.show()
TOP_LIME_FEATURES = [
    next((f for f in FEATURES if feat.startswith(f)), feat.split('<=')[0].strip())
    for feat, _ in exp_lime.as_list(label=CLASE_SHAP)[:3]
]
print(f"Top-3 LIME: {TOP_LIME_FEATURES}")


In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `TOP_LIME_FEATURES`
r = []
try:
    r = verificar_lime_local(INDEX_CASO, len(X_test), TOP_LIME_FEATURES)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — LIME local', r)


## Pregunta 11 — PDP y SHAP dependence (efecto marginal)

### 📘 Subpreguntas
- **10.a** ¿Qué muestra el PDP frente al dependence plot de SHAP?
- **10.b** ¿La relación de `FEATURE_PDP` con el daño es monótona?


#### ✅ Reflexión 11

PDP = efecto marginal promedio; dependence captura interacciones.


In [ ]:
# --- PRE-ESCRITO: PDP para Strain ---
idx_strain = FEATURES.index('Strain (με)')
fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(
    modelo, X_test, [idx_strain], feature_names=FEATURES, target=2, ax=ax,
)
ax.set_title('PDP — Strain (με)')
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
FEATURE_PDP = "Temp (°C)"
idx_pdp = FEATURES.index(FEATURE_PDP)
fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(modelo, X_test, [idx_pdp], feature_names=FEATURES, target=CLASE_SHAP, ax=ax)
ax.set_title(f'PDP — {FEATURE_PDP}')
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 4))
shap.dependence_plot(idx_pdp, shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
plt.title(f'SHAP dependence — {FEATURE_PDP}')
plt.tight_layout(); plt.show()


In [ ]:
# --- Autoevaluación 11 ---
# Requiere (celda «Aquí coloca tu solución»): `FEATURE_PDP`
r = []
try:
    r = verificar_pdp(FEATURE_PDP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('11 — PDP y dependence', r)


## Cierre — respuestas sugeridas

- **Strain:** domina importancia y SHAP — coherente con extensómetro en daño.
- **Obra:** inspección visual + histórico; xAI no sustituye normativa.
- **SHAP vs LIME:** suelen coincidir en Strain; LIME puede variar por muestreo local.
- **Límite:** correlación entre sensores; xAI explica el modelo, no causalidad física.
